<a href="https://colab.research.google.com/github/2nithin2/ebook2audiobook/blob/main/Notebooks/colab_ebook2audiobook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup: Clean Wipe and Fresh Clone

In [6]:
import os

!rm -rf /content/ebook2audiobook
!git clone --depth=1 https://github.com/DrewThomasson/ebook2audiobook.git /content/ebook2audiobook

print(f"✅ ebook2audiobook cloned to {os.environ.get('SCRIPT_DIR', '/content/ebook2audiobook')}")

Cloning into '/content/ebook2audiobook'...
remote: Enumerating objects: 788, done.
remote: Counting objects: 100% (788/788), done.
remote: Compressing objects: 100% (556/556), done.
remote: Total 788 (delta 186), reused 658 (delta 162), pack-reused 0 (from 0)
Receiving objects: 100% (788/788), 31.03 MiB | 30.85 MiB/s, done.
Resolving deltas: 100% (186/186), done.
✅ ebook2audiobook cloned to /content/ebook2audiobook


## Welcome to the ebook2audiobook Google Colab!
## Features
- 🔧 **TTS Engines supported**: XTTSv2, Bark, Fairseq, VITS, Tacotron2, Tortoise, GlowTTS, YourTTS- 📚 **Convert multiple file formats**: .epub, .mobi, .azw3, .fb2, .lrf, .rb, .snb, .tcr, .pdf, .txt, .rtf, .doc, .docx, .html, .odt, .azw, .tiff, .tif, .png, .jpg, .jpeg, .bmp
- 🔍 **OCR scanning** for files with text pages as images
- 🔊 **High-quality text-to-speech** from near realtime to near real voice
- 🗣️ **Optional voice cloning** using your own voice file
- 🌐 **Supports 1158 languages** ([supported languages list](https://dl.fbaipublicfiles.com/mms/tts/all-tts-languages.html))
- 💻 **Low-resource friendly** — runs on **2 GB RAM / 1 GB VRAM (minimum)**
- 🎵 **Audiobook output formats**: mono or stereo aac, flac, mp3, m4b, m4a, mp4, mov, ogg, wav, webm
- 🧠 **SML tags supported** — fine-grained control of breaks, pauses, voice switching and more ([see below](#sml-tags-available))
- 🧩 **Optional custom model** using your own trained model (XTTSv2 only, other on request)
- 🎛️ **Fine-tuned preset models** trained by the E2A Team<br/>
     <i>(Contact us if you need additional fine-tuned models, or if you'd like to share yours to the official preset list)</i>
## Want to run locally for free? ⬇
## [Check out the ebook2audiobook github!](https://github.com/DrewThomasson/ebook2audiobook)

In [5]:
# @title 🚀 Run ebook2audiobook!

import os
import subprocess
import time
import shutil
import sysconfig

# Emojis for logs
CHECK_MARK = "✅"
CROSS_MARK = "❌"

SCRIPT_DIR = "/content/ebook2audiobook"

# ── Environment variables (mirrors the bash script) ──────────────────────────
os.environ["PYTHONUTF8"]        = "1"
os.environ["PYTHONIOENCODING"]  = "utf-8"
os.environ["TTS_CACHE"]         = f"{SCRIPT_DIR}/models"
os.environ["TESSDATA_PREFIX"]   = f"{SCRIPT_DIR}/models/tessdata"
os.environ["TMPDIR"]            = f"{SCRIPT_DIR}/tmp"

def display_loading_bar(total_steps):
    print("\n---  LOADING...  Total steps:", total_steps, " ---")

def update_progress(step, total_steps):
    bar_length = 20
    progress_percent = int((step / total_steps) * 100)
    progress_filled = int(bar_length * step / total_steps)
    bar = '=' * progress_filled + '>' + ' ' * max(bar_length - progress_filled - 1, 0)
    print(f"---  PROGRESS:  [{bar}] {progress_percent}% ({step}/{total_steps}) ---")

def run_command_with_log(command, description, step_progress, total_step_commands):
    """Runs a shell command and logs progress, outcome and duration."""
    print(f"\n{step_progress}/{total_step_commands}: {description}...")
    start_time = time.time()
    try:
        process = subprocess.Popen(command, shell=True,
                                   stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        stdout, stderr = process.communicate()
        duration = f"{time.time() - start_time:.2f}"
        if process.returncode != 0:
            print(f"{CROSS_MARK} Command failed: {description} (Took {duration}s)")
            print(f"   Command: {command}")
            print(f"   Error Output:\n{stderr.decode()}")
            return False
        else:
            update_progress(step_progress, total_step_commands)
            print(f"{CHECK_MARK} {step_progress}/{total_step_commands} completed: {description} (Took {duration}s)")
            return True
    except Exception as e:
        duration = f"{time.time() - start_time:.2f}"
        print(f"{CROSS_MARK} Error during: {description} (Took {duration}s) — {e}")
        return False


# ── Step 1 : OS-level packages ────────────────────────────────────────────────
# Mirrors HOST_PROGRAMS from the bash script + Calibre's special installer
os_install_commands = [
    ("apt-get update -qq",
     "Update package lists"),
    ("apt-get install -y -qq libxcb-cursor0 libegl1 libopengl0",
     "Install Calibre display libraries (libxcb-cursor0, libegl1, libopengl0)"),
    ("sudo -v && wget -nv -O- https://download.calibre-ebook.com/linux-installer.sh | sudo sh /dev/stdin",
     "Download & install Calibre"),
    ("apt-get install -y -qq ffmpeg",
     "Install ffmpeg"),
    ("apt-get install -y -qq mediainfo",
     "Install mediainfo"),
    ("apt-get install -y -qq nodejs",
     "Install nodejs"),
    ("apt-get install -y -qq espeak-ng",
     "Install espeak-ng"),
    ("apt-get install -y -qq sox",
     "Install sox"),
    ("apt-get install -y -qq tesseract-ocr tesseract-ocr-eng",
     "Install Tesseract OCR + English language pack"),
    # mecab + unidic (for Japanese support, same as old notebook)
    ("apt-get install -y -qq mecab libmecab-dev mecab-ipadic-utf8",
     "Install mecab (Japanese text analysis)"),
    # Rust is required by some pip packages (e.g. tokenizers)
    ("curl -fsSL https://sh.rustup.rs | sh -s -- -y --quiet && "
     "echo 'source $HOME/.cargo/env' >> ~/.bashrc",
     "Install Rust (required by some Python packages)"),
]

# ── Step 2 : Git clone ────────────────────────────────────────────────────────
git_commands = [
    ("git clone --depth=1 https://github.com/DrewThomasson/ebook2audiobook.git /content/ebook2audiobook",
     "Git clone ebook2audiobook"),
]

# ── Step 3 : Python packages  (mirrors install_python_packages()) ─────────────
# llvmlite/numba must be installed first as pre-built binaries;
# then the rest of requirements.txt is installed line-by-line.
# unidic download closes out the step.
pip_commands = [
    ("pip install -q --upgrade pip setuptools wheel",
     "Upgrade pip / setuptools / wheel"),
    ("pip install -q --upgrade llvmlite numba --only-binary=:all:",
     "Install llvmlite & numba (pre-built binaries only)"),
    ("pip install -q --upgrade --no-cache-dir "
     "-r /content/ebook2audiobook/requirements.txt",
     "Install Python requirements from requirements.txt"),
    ("python -m unidic download",
     "Download Unidic dictionary data"),
]


# ── Helper: sitecustomize hook (mirrors check_sitecustomized()) ───────────────
def install_sitecustomize():
    src = f"{SCRIPT_DIR}/components/sitecustomize.py"
    dst_dir = sysconfig.get_paths()["purelib"]
    dst = os.path.join(dst_dir, "sitecustomize.py")
    if not os.path.exists(src):
        print(f"{CROSS_MARK} sitecustomize.py source not found at {src}, skipping.")
        return False
    if not os.path.exists(dst) or os.path.getmtime(src) > os.path.getmtime(dst):
        shutil.copy2(src, dst)
        print(f"{CHECK_MARK} Installed sitecustomize.py hook → {dst}")
    else:
        print(f"{CHECK_MARK} sitecustomize.py already up-to-date.")
    return True


# ════════════════════════════════════════════════════════════════════════════
# EXECUTION
# ════════════════════════════════════════════════════════════════════════════

# --- Step 1: OS packages ---
print("\n--- Step 1: OS-Level Installations ---")
print("Installs system packages required by the bash script. (~3-5 min)")
display_loading_bar(len(os_install_commands))
step1_ok = True
for i, (cmd, desc) in enumerate(os_install_commands):
    if not run_command_with_log(cmd, desc, i + 1, len(os_install_commands)):
        step1_ok = False
        break

# Activate Rust for the current session after installation
cargo_env = os.path.expanduser("~/.cargo/env")
if os.path.exists(cargo_env):
    os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ.get("PATH", "")

if step1_ok:
    print(f"\n{CHECK_MARK} Step 1: OS-Level Installations — Completed Successfully!")
else:
    print(f"\n{CROSS_MARK} Step 1: OS-Level Installations — Failed. See errors above.")


# --- Step 2: Git Clone ---
print("\n--- Step 2: Git Clone ---")
display_loading_bar(len(git_commands))
step2_ok = True
for i, (cmd, desc) in enumerate(git_commands):
    if not run_command_with_log(cmd, desc, i + 1, len(git_commands)):
        step2_ok = False
        break

if step2_ok:
    print(f"\n{CHECK_MARK} Step 2: Git Clone — Completed Successfully!")
else:
    print(f"\n{CROSS_MARK} Step 2: Git Clone — Failed. See errors above.")


# --- Step 3: Python packages ---
print("\n--- Step 3: Python Package Installation ---")
print("Mirrors install_python_packages() from the bash script. (~3-5 min)")
display_loading_bar(len(pip_commands))
step3_ok = True
for i, (cmd, desc) in enumerate(pip_commands):
    if not run_command_with_log(cmd, desc, i + 1, len(pip_commands)):
        step3_ok = False
        break

if step3_ok:
    print(f"\n{CHECK_MARK} Step 3: Python Package Installation — Completed Successfully!")
else:
    print(f"\n{CROSS_MARK} Step 3: Python Package Installation — Failed. See errors above.")


# --- Step 4: sitecustomize hook ---
print("\n--- Step 4: Installing sitecustomize.py hook ---")
install_sitecustomize()


# --- Step 5: Create required directories ---
print("\n--- Step 5: Creating required directories ---")
for d in ["models", "models/tessdata", "tmp", "run", "audiobooks", "ebooks", "voices"]:
    os.makedirs(f"{SCRIPT_DIR}/{d}", exist_ok=True)
os.chmod(f"{SCRIPT_DIR}/tmp", 0o777)
print(f"{CHECK_MARK} Directories ready.")


# --- Step 6: Run App ---
print("\n--- Step 6: Launch ebook2audiobook ---")
print("Starting the Gradio web interface with a public share link...")
try:
    # script_mode=native matches the NATIVE branch of the bash script;
    # --share creates the public Gradio tunnel link.
    get_ipython().system(
        "cd /content/ebook2audiobook && "
        "python -u app.py --script_mode native --share"
    )
except Exception as e:
    print(f"{CROSS_MARK} Error starting app.py: {e}")

print("\n--- All Steps Completed ---")
print("Check the output above for the public Gradio URL.")



--- Step 1: OS-Level Installations ---
Installs system packages required by the bash script. (~3-5 min)

---  LOADING...  Total steps: 11  ---

1/11: Update package lists...
---  PROGRESS:  [=>                  ] 9% (1/11) ---
✅ 1/11 completed: Update package lists (Took 3.15s)

2/11: Install Calibre display libraries (libxcb-cursor0, libegl1, libopengl0)...
---  PROGRESS:  [===>                ] 18% (2/11) ---
✅ 2/11 completed: Install Calibre display libraries (libxcb-cursor0, libegl1, libopengl0) (Took 3.43s)

3/11: Download & install Calibre...
---  PROGRESS:  [=====>              ] 27% (3/11) ---
✅ 3/11 completed: Download & install Calibre (Took 22.63s)

4/11: Install ffmpeg...
---  PROGRESS:  [=======>            ] 36% (4/11) ---
✅ 4/11 completed: Install ffmpeg (Took 3.63s)

5/11: Install mediainfo...
---  PROGRESS:  [=========>          ] 45% (5/11) ---
✅ 5/11 completed: Install mediainfo (Took 2.75s)

6/11: Install nodejs...
---  PROGRESS:  [==========>         ] 54% (6/11) 

## Python Package Installation

In [7]:
import os

os.chdir('/content/ebook2audiobook')
!pip install -q --upgrade pip setuptools wheel
!pip install -q --upgrade --no-cache-dir -r requirements.txt

print("--- Step: unidic download ---")
!python -m unidic download
print("✅ Python packages installed and Unidic data downloaded.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-adk 2.3.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.43.0 which is incompatible.
google-adk 2.3.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.43.0 which is incompatible.
google-adk 2.3.0 requires pydantic<3,>=2.12, but you have pydantic 2.11.10 which is incompatible.
google-adk 2.3.0 requires starlette<2,>=1.0.1, but you have starlette 0.52.1 which is incompatible.
python-fasthtml 0.14.3 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but y

## Fix for `torch`/`torchaudio` Mismatch

In [11]:
print("--- Step: Fixing torch/torchaudio mismatch with explicit pinning ---")
!pip uninstall -y torch torchaudio
!pip install torch==2.6.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu128
print("✅ torch and torchaudio reinstalled and pinned to 2.6.0.")

--- Step: Fixing torch/torchaudio mismatch ---
Found existing installation: torch 2.9.0+cu128
Uninstalling torch-2.9.0+cu128:
  Successfully uninstalled torch-2.9.0+cu128
Found existing installation: torchaudio 2.9.0+cu128
Uninstalling torchaudio-2.9.0+cu128:
  Successfully uninstalled torchaudio-2.9.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 107.1 MB/s  0:00:07
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 40.9 MB/s  0:00:06
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 134.4 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 66.3 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.8/930.8 kB 24.6 MB/s  0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.5.0
    Uninstalling triton-3.5.0:
      Successfully uninstalled triton-3.5.0
  Attempting uninstall: setuptools
    Found existing installation: setuptools 82

✅ torch and torchaudio reinstalled.


## Optional Fallback: Pinning `torch` and `torchaudio` Versions

In [12]:
# If the above fix doesn't work, uncomment and run these lines to pin specific versions.
# !pip uninstall -y torch torchaudio
# !pip install torch==2.6.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu128
# print("✅ torch and torchaudio pinned to specific versions.")

## Verify `torch` and `torchaudio` Versions

In [13]:
import torch, torchaudio
print(f"torch version: {torch.__version__}")
print(f"torchaudio version: {torchaudio.__version__}")

AttributeError: module 'torch' has no attribute '__version__'

## Preemptive CUDA Runtime Fix

In [8]:
print("--- Step: CUDA Runtime Installation ---")
!pip install -q nvidia-cuda-runtime-cu12
print("✅ nvidia-cuda-runtime-cu12 installed.")

--- Step: CUDA Runtime Installation ---
✅ nvidia-cuda-runtime-cu12 installed.


## Optional Fallback to CPU-only onnxruntime (Run only if `libcudart.so.13` error persists)

In [9]:
# If you still encounter `libcudart.so.13` errors, uncomment and run the following cells to force CPU-only onnxruntime.
# !pip uninstall -y onnxruntime onnxruntime-gpu
# !pip install -q onnxruntime
# print("✅ onnxruntime reinstalled for CPU-only mode.")

## Launch ebook2audiobook Gradio Interface

In [10]:
import os
import shutil
import sysconfig

SCRIPT_DIR = "/content/ebook2audiobook"

os.chdir(SCRIPT_DIR)

# --- Create required directories ---
print("--- Step: Creating required directories ---")
for d in ["models", "models/tessdata", "tmp", "run", "audiobooks", "ebooks", "voices"]:
    os.makedirs(f"{SCRIPT_DIR}/{d}", exist_ok=True)
os.chmod(f"{SCRIPT_DIR}/tmp", 0o777)
print("✅ Directories ready.")

# --- Install sitecustomize.py hook ---
print("--- Step: Installing sitecustomize.py hook ---")
src = f"{SCRIPT_DIR}/components/sitecustomize.py"
dst_dir = sysconfig.get_paths()["purelib"]
dst = os.path.join(dst_dir, "sitecustomize.py")
if not os.path.exists(src):
    print(f"❌ sitecustomize.py source not found at {src}, skipping.")
else:
    if not os.path.exists(dst) or os.path.getmtime(src) > os.path.getmtime(dst):
        shutil.copy2(src, dst)
        print(f"✅ Installed sitecustomize.py hook → {dst}")
    else:
        print(f"✅ sitecustomize.py already up-to-date.")

# --- Launch App ---
print("\n--- Launch ebook2audiobook ---")
print("Starting the Gradio web interface with a public share link...")

# Ensure environment variables are set before launching the app
os.environ["PYTHONUTF8"]        = "1"
os.environ["PYTHONIOENCODING"]  = "utf-8"
os.environ["TTS_CACHE"]         = f"{SCRIPT_DIR}/models"
os.environ["TESSDATA_PREFIX"]   = f"{SCRIPT_DIR}/models/tessdata"
os.environ["TMPDIR"]            = f"{SCRIPT_DIR}/tmp"

# Activate Rust for the current session (if not already done by OS install cell)
cargo_env = os.path.expanduser("~/.cargo/env")
if os.path.exists(cargo_env):
    os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ.get("PATH", "")

!python app.py --share

--- Step: Creating required directories ---
✅ Directories ready.
--- Step: Installing sitecustomize.py hook ---
✅ Installed sitecustomize.py hook → /usr/local/lib/python3.12/dist-packages/sitecustomize.py

--- Launch ebook2audiobook ---
Starting the Gradio web interface with a public share link...
v26.6.30 native mode
---> Hardware detected: {'name': 'cuda', 'os': 'manylinux_2_28', 'arch': 'x86_64', 'pyvenv': [3, 12], 'tag': 'cu128', 'note': ''}
Found existing installation: onnxruntime 1.27.0
Uninstalling onnxruntime-1.27.0:
  Successfully uninstalled onnxruntime-1.27.0


IPs available for connection:
['127.0.0.1', '::1', '172.28.0.12']
Note: 0.0.0.0 is not the IP to connect. Instead use an IP above to connect and port 7860
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://b84c8810558b63edf3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to 